# Model Training - Candidate Regression Models for Achievement Ratio

Trains and compares seven regressors (Linear, Ridge, Lasso, Random Forest, Gradient Boosting, Extra Trees, XGBoost) on `training_ready.csv` using a chronological train/test split, and selects the best by R².


## 1. Imports
- Models: Linear Regression, Ridge, Lasso, Random Forest, Gradient Boosting, Extra Trees, XGBoost. 
- Metrics: MAE, MSE, RMSE, R².


In [21]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor

# Models
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    ExtraTreesRegressor
)

# Metrics
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

## 2. Load Training-Ready Dataset
Output of the preprocessing/encoding notebook (`training_ready.csv`).


In [22]:
# 1. Load dataset

df = pd.read_csv("training_ready.csv")

print("Dataset shape:", df.shape)

Dataset shape: (9446, 31)


## 3. Define Features (X) and Target (y)
Target: `achievement_ratio`.


In [23]:
# 2. Define target

target = "actual_qty"

X = df.drop(columns=[target])
y = df[target]

## 4. Chronological Train/Test Split 
Replaces the earlier random `train_test_split`. Takes the first 80% of rows as training and the last 20% as test, by position - train on earlier weeks, test on the most recent weeks, no shuffling.

In [24]:
# 80% training, 20% testing
split_index = int(len(X) * 0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

Confirm the resulting split sizes.


In [25]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((7556, 30), (1890, 30), (7556,), (1890,))

## 5. Feature & Target Scaling

`planned_qty` and `no_of_workers` are standardized (fit on train only, applied to test - correct direction, no leakage here).

> ⚠️ **Still flagged — not yet fixed.** `achievement_ratio` (the target) is standardized via `y_scaler`. This does not affect R² (scale-invariant), but **MAE and RMSE in the results table below remain in standardized z-score units, not achievement-ratio units.** Either drop the target scaling entirely (tree models don't need it, and the linear models are affected only marginally), or inverse-transform predictions back to the original scale before computing MAE/RMSE.

In [26]:
from sklearn.preprocessing import StandardScaler
import joblib

X_scaler = StandardScaler()
y_scaler = StandardScaler()

X_train[["planned_qty", "no_of_workers"]] = X_scaler.fit_transform(X_train[["planned_qty", "no_of_workers"]])
X_test[["planned_qty", "no_of_workers"]] = X_scaler.transform(X_test[["planned_qty", "no_of_workers"]])

y_train = y_scaler.fit_transform(y_train.values.reshape(-1, 1))
y_test = y_scaler.transform(y_test.values.reshape(-1, 1))

joblib.dump(X_scaler, "X_scaler.joblib")
joblib.dump(y_scaler, "y_scaler.joblib")

/var/folders/7v/gsp_3mnd53bbrxcw6242zd640000gn/T/ipykernel_16369/3947794886.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train[["planned_qty", "no_of_workers"]] = X_scaler.fit_transform(X_train[["planned_qty", "no_of_workers"]])
/var/folders/7v/gsp_3mnd53bbrxcw6242zd640000gn/T/ipykernel_16369/3947794886.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_test[["planned_qty", "no_of_workers"]] = X_scaler.transform(X_test[["planned_qty", "no_of_workers"]])


['y_scaler.joblib']

## 6. Define Candidate Models
- Linear, Ridge, and Lasso are wrapped in their own `StandardScaler` pipelines (in addition to the manual scaling above - redundant but harmless). 
- Tree-based models are left unscaled, which is correct since they don't require it.


In [27]:
# 4. Define models

models = {

    "Linear Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ]),

    "Ridge Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=1.0))
    ]),

    "Lasso Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", Lasso(alpha=0.001))
    ]),

    "Random Forest": RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    ),

    "Extra Trees": ExtraTreesRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ),
    
    "XGBoost": XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
    )
}

## 7. Train & Evaluate Every Model
Fits each model on the chronological training split and scores it on the held-out (future) test split.


In [28]:
# 5. Train and evaluate all models

results = []

for name, model in models.items():

    print(f"\nTraining: {name}")

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Metrics
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    results.append({
        "Model": name,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2
    })


Training: Linear Regression

Training: Ridge Regression

Training: Lasso Regression

Training: Random Forest


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)



Training: Gradient Boosting


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/ensemble/_gb.py:672: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)  # TODO: Is this still required?



Training: Extra Trees


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)



Training: XGBoost


## 8. Compare Results
Sorted by R² (descending). R² values are valid and comparable; **MAE/RMSE here are still in scaled units** until Section 5's flagged issue is resolved.


In [29]:
# 6. Compare results

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="R2",
    ascending=False
)

print("\n====================================")
print("MODEL COMPARISON")
print("====================================")

print(results_df.to_string(index=False))


MODEL COMPARISON
            Model      MAE      MSE     RMSE       R2
Gradient Boosting 0.030747 0.015030 0.122595 0.989553
      Extra Trees 0.032844 0.015576 0.124803 0.989174
    Random Forest 0.032096 0.017038 0.130529 0.988157
 Lasso Regression 0.035337 0.018883 0.137414 0.986875
 Ridge Regression 0.036636 0.018945 0.137640 0.986832
Linear Regression 0.036628 0.018950 0.137657 0.986828
          XGBoost 0.038645 0.026957 0.164187 0.981262


## 9. Select the Best Model
Picks the top row of the comparison table by R².


In [30]:
# 7. Select best model based on R²

best_model_name = results_df.iloc[0]["Model"]

print("\n====================================")
print("BEST MODEL")
print("====================================")

print("Best model:", best_model_name)
print("R²:", results_df.iloc[0]["R2"])
print("RMSE:", results_df.iloc[0]["RMSE"])
print("MAE:", results_df.iloc[0]["MAE"])


BEST MODEL
Best model: Gradient Boosting
R²: 0.989553178240464
RMSE: 0.12259494142934284
MAE: 0.030746517598090786


In [31]:
model_1 = models["Linear Regression"]
model_2 = models["Ridge Regression"]
model_3 = models["Lasso Regression"]
model_4 = models["Random Forest"]
model_5 = models["Gradient Boosting"]
model_6 = models["Extra Trees"]
model_7 = models["XGBoost"]

In [32]:
train_pred = model_1.predict(X_train)
test_pred = model_1.predict(X_test)

train_r2 = r2_score(y_train, train_pred)
test_r2 = r2_score(y_test, test_pred)

train_mae = mean_absolute_error(y_train, train_pred)
test_mae = mean_absolute_error(y_test, test_pred)

print("Train R²:", train_r2)
print("Test R²:", test_r2)

print("Train MAE:", train_mae)
print("Test MAE:", test_mae)

Train R²: 0.987471534738679
Test R²: 0.9868284260031325
Train MAE: 0.03193485030161019
Test MAE: 0.036628004334919154


In [33]:
train_pred = model_2.predict(X_train)
test_pred = model_2.predict(X_test)

train_r2 = r2_score(y_train, train_pred)
test_r2 = r2_score(y_test, test_pred)

train_mae = mean_absolute_error(y_train, train_pred)
test_mae = mean_absolute_error(y_test, test_pred)

print("Train R²:", train_r2)
print("Test R²:", test_r2)

print("Train MAE:", train_mae)
print("Test MAE:", test_mae)

Train R²: 0.9874715173515075
Test R²: 0.9868317422394399
Train MAE: 0.031940564446292015
Test MAE: 0.03663561274458473


In [34]:
train_pred = model_3.predict(X_train)
test_pred = model_3.predict(X_test)

train_r2 = r2_score(y_train, train_pred)
test_r2 = r2_score(y_test, test_pred)

train_mae = mean_absolute_error(y_train, train_pred)
test_mae = mean_absolute_error(y_test, test_pred)

print("Train R²:", train_r2)
print("Test R²:", test_r2)

print("Train MAE:", train_mae)
print("Test MAE:", test_mae)

Train R²: 0.9874500612077135
Test R²: 0.9868749314935923
Train MAE: 0.03095706733172666
Test MAE: 0.03533668041064286


In [35]:
train_pred = model_4.predict(X_train)
test_pred = model_4.predict(X_test)

train_r2 = r2_score(y_train, train_pred)
test_r2 = r2_score(y_test, test_pred)

train_mae = mean_absolute_error(y_train, train_pred)
test_mae = mean_absolute_error(y_test, test_pred)

print("Train R²:", train_r2)
print("Test R²:", test_r2)

print("Train MAE:", train_mae)
print("Test MAE:", test_mae)

Train R²: 0.9985018362192539
Test R²: 0.9881572253087341
Train MAE: 0.009842730115652879
Test MAE: 0.032096395731438826


In [36]:
train_pred = model_5.predict(X_train)
test_pred = model_5.predict(X_test)

train_r2 = r2_score(y_train, train_pred)
test_r2 = r2_score(y_test, test_pred)

train_mae = mean_absolute_error(y_train, train_pred)
test_mae = mean_absolute_error(y_test, test_pred)

print("Train R²:", train_r2)
print("Test R²:", test_r2)

print("Train MAE:", train_mae)
print("Test MAE:", test_mae)

Train R²: 0.9985120233056369
Test R²: 0.989553178240464
Train MAE: 0.018281689407129688
Test MAE: 0.030746517598090786


In [37]:
train_pred = model_6.predict(X_train)
test_pred = model_6.predict(X_test)

train_r2 = r2_score(y_train, train_pred)
test_r2 = r2_score(y_test, test_pred)

train_mae = mean_absolute_error(y_train, train_pred)
test_mae = mean_absolute_error(y_test, test_pred)

print("Train R²:", train_r2)
print("Test R²:", test_r2)

print("Train MAE:", train_mae)
print("Test MAE:", test_mae)

Train R²: 0.9999993971719675
Test R²: 0.9891735566188248
Train MAE: 5.765852640426234e-05
Test MAE: 0.03284354848935069


In [38]:
train_pred = model_7.predict(X_train)
test_pred = model_7.predict(X_test)

train_r2 = r2_score(y_train, train_pred)
test_r2 = r2_score(y_test, test_pred)

train_mae = mean_absolute_error(y_train, train_pred)
test_mae = mean_absolute_error(y_test, test_pred)

print("Train R²:", train_r2)
print("Test R²:", test_r2)

print("Train MAE:", train_mae)
print("Test MAE:", test_mae)

Train R²: 0.9991045527177782
Test R²: 0.9812623210334024
Train MAE: 0.017150504115876573
Test MAE: 0.03864483928596621


## 10. Save Best Model

In [39]:
best_model = models["Gradient Boosting"]

joblib.dump(best_model, "gradient_boosting_model.joblib")

print("Model saved successfully!")

Model saved successfully!
